In [ ]:
# ==============================================================\n
# R0 — Runtime / DuckDB lock-safe bootstrap\n
# ==============================================================\n
from pathlib import Path\n
import duckdb\n
import tomllib\n
import atexit\n
\n
PROJECT_ROOT = Path.cwd()\n
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"\n
\n
if CONFIG_TOML_PATH.exists():\n
    with open(CONFIG_TOML_PATH, "rb") as f:\n
        SHARED_CONFIG = tomllib.load(f)\n
    print("Loaded shared config:", CONFIG_TOML_PATH)\n
else:\n
    SHARED_CONFIG = {\n
        "paths": {\n
            "data_dir": "content",\n
            "output_dir": "outputs_benchmark_survival",\n
            "tables_subdir": "tables",\n
            "metadata_subdir": "metadata",\n
            "data_output_subdir": "data",\n
            "duckdb_filename": "benchmark_survival.duckdb",\n
        },\n
        "benchmark": {\n
            "seed": 42,\n
            "test_size": 0.30,\n
            "early_window_weeks": 4,\n
            "main_enrollment_window_weeks": 4,\n
        },\n
    }\n
    print("Shared config TOML not found. Using defaults in-memory.")\n
\n
paths_cfg = SHARED_CONFIG.get("paths", {})\n
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})\n
\n
def _resolve_project_path(raw_path: str) -> Path:\n
    p = Path(raw_path)\n
    return p if p.is_absolute() else PROJECT_ROOT / p\n
\n
DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))\n
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))\n
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")\n
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")\n
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")\n
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")\n
\n
for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:\n
    p.mkdir(parents=True, exist_ok=True)\n
\n
if "con" in globals():\n
    try:\n
        con.close()\n
        print("Closed previous DuckDB connection before reconnecting.")\n
    except Exception as _close_err:\n
        print(f"Warning while closing previous DuckDB connection: {_close_err}")\n
\n
con = duckdb.connect(str(DUCKDB_PATH))\n
\n
def _close_duckdb_connection() -> None:\n
    if "con" in globals():\n
        try:\n
            con.close()\n
            print("DuckDB connection closed.")\n
        except Exception:\n
            pass\n
\n
if "_duckdb_close_registered" not in globals():\n
    atexit.register(_close_duckdb_connection)\n
    _duckdb_close_registered = True\n
\n
print("Runtime context ready.")\n
print("- OUTPUT_DIR  :", OUTPUT_DIR)\n
print("- DUCKDB_PATH :", DUCKDB_PATH)\n

# B - Feature engineering and benchmark input construction

This section transforms the canonical survival backbone into the model-ready benchmark inputs.

It covers:
- person-period expansion
- weekly behavioral feature aggregation
- temporal feature engineering
- enrollment-level model tables
- canonical benchmark configuration
- model-specific input views
- window-based comparable features

The outputs of this section feed the split, preprocessing, and modeling steps in the following sections.

## B0 — Runtime Rehydration (Standalone Execution)

This setup cell rebuilds the required runtime context for section B when running this notebook in an isolated kernel.

It restores:
- project paths
- output directories
- DuckDB connection
- dependency check on upstream table produced by section A (`enrollment_survival_ready`)

In [2]:
from pathlib import Path
import duckdb
import tomllib

print("\n" + "=" * 70)
print("B0 — Runtime Rehydration (Standalone Execution)")
print("=" * 70)

PROJECT_ROOT = Path.cwd()
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"

if CONFIG_TOML_PATH.exists():
    with open(CONFIG_TOML_PATH, "rb") as f:
        SHARED_CONFIG = tomllib.load(f)
    print("Loaded shared config:", CONFIG_TOML_PATH)
else:
    SHARED_CONFIG = {
        "paths": {
            "data_dir": "content",
            "output_dir": "outputs_benchmark_survival",
            "tables_subdir": "tables",
            "metadata_subdir": "metadata",
            "data_output_subdir": "data",
            "duckdb_filename": "benchmark_survival.duckdb",
        },
        "benchmark": {
            "seed": 42,
            "test_size": 0.30,
            "early_window_weeks": 4,
            "main_enrollment_window_weeks": 4,
        },
    }
    print("Shared config TOML not found. Using defaults in-memory.")

paths_cfg = SHARED_CONFIG.get("paths", {})
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})


def _resolve_project_path(raw_path: str) -> Path:
    p = Path(raw_path)
    return p if p.is_absolute() else PROJECT_ROOT / p


DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")

for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DUCKDB_PATH))

available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())

# Rehydrate core upstream table from persisted A outputs if needed.
if "enrollment_survival_ready" not in available_tables:
    esr_parquet = DATA_OUTPUT_DIR / "enrollment_survival_ready.parquet"
    esr_csv = DATA_OUTPUT_DIR / "enrollment_survival_ready.csv"

    if esr_parquet.exists():
        con.execute("DROP TABLE IF EXISTS enrollment_survival_ready")
        con.execute(
            f"CREATE TABLE enrollment_survival_ready AS SELECT * FROM read_parquet('{esr_parquet.as_posix()}')"
        )
        print("Rehydrated enrollment_survival_ready from:", esr_parquet)
    elif esr_csv.exists():
        con.execute("DROP TABLE IF EXISTS enrollment_survival_ready")
        con.execute(
            f"CREATE TABLE enrollment_survival_ready AS SELECT * FROM read_csv_auto('{esr_csv.as_posix()}', HEADER=TRUE)"
        )
        print("Rehydrated enrollment_survival_ready from:", esr_csv)

# Re-register studentVle from raw CSV if missing (needed by B2).
available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())
if "studentVle" not in available_tables:
    student_vle_csv = DATA_DIR / "studentVle.csv"
    if student_vle_csv.exists():
        con.execute("DROP VIEW IF EXISTS studentVle")
        con.execute(
            f"CREATE VIEW studentVle AS SELECT * FROM read_csv_auto('{student_vle_csv.as_posix()}', HEADER=TRUE)"
        )
        print("Registered studentVle from:", student_vle_csv)

available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())
required_tables = ["enrollment_survival_ready", "studentVle"]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required upstream table(s): "
        + ", ".join(missing_tables)
        + f". Run dropout_bench_v3_A_foundation.ipynb (at least up to A5) or ensure source files exist in {DATA_DIR}/ and {DATA_OUTPUT_DIR}/."
    )

print("Runtime context ready.")
print("- PROJECT_ROOT:", PROJECT_ROOT)
print("- OUTPUT_DIR  :", OUTPUT_DIR)
print("- DUCKDB_PATH :", DUCKDB_PATH)
print("- Shared benchmark seed:", SHARED_BENCHMARK_CONFIG.get("seed", 42))
print("- DuckDB tables currently available:", len(available_tables))


B0 — Runtime Rehydration (Standalone Execution)
Loaded shared config: /workspaces/survival_dropout_benchmark/benchmark_shared_config.toml
Runtime context ready.
- PROJECT_ROOT: /workspaces/survival_dropout_benchmark
- OUTPUT_DIR  : /workspaces/survival_dropout_benchmark/outputs_benchmark_survival
- DUCKDB_PATH : /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/benchmark_survival.duckdb
- Shared benchmark seed: 42
- DuckDB tables currently available: 22


## B1 — Build Minimal Person-Period Skeleton


In [3]:
# ==============================================================
# B1 — Build Minimal Person-Period Skeleton
# --------------------------------------------------------------
# Purpose:
#   Expand the enrollment-level survival-ready table into a
#   weekly person-period skeleton, preserving one row per
#   enrollment-week and creating the discrete-time event target.
#
#   This revised version explicitly clamps negative t_final_week
#   values to zero before expansion so that no enrollment is lost.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: person_period_min
#   - person_period_min.parquet
#   - person_period_min.csv
#   - table_person_period_min_audit.csv
#   - table_person_period_event_check.csv
#   - table_person_period_week_distribution.csv
# ==============================================================

print("\n" + "=" * 70)
print("B1 — Build Minimal Person-Period Skeleton")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUTPUT_DIR / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Rebuild person-period skeleton
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_min")

con.execute("""
CREATE TABLE person_period_min AS
WITH base AS (
    SELECT
        esr.*,
        GREATEST(esr.t_final_week, 0) AS t_final_week_clamped
    FROM enrollment_survival_ready AS esr
),
expanded AS (
    SELECT
        b.id_student,
        b.code_module,
        b.code_presentation,
        b.gender,
        b.region,
        b.highest_education,
        b.imd_band,
        b.age_band,
        b.num_of_prev_attempts,
        b.studied_credits,
        b.disability,
        b.final_result,
        b.date_registration,
        b.date_unregistration,
        b.is_withdrawn,
        b.has_valid_unregistration_date,
        b.event_observed,
        b.is_withdrawn_without_valid_unregistration,
        b.has_any_vle_activity,
        b.max_vle_day,
        b.n_vle_rows,
        b.total_clicks_all_time,
        b.t_event_week,
        b.t_last_obs_week,
        b.t_final_week,
        b.t_final_week_clamped,
        b.used_zero_week_fallback_for_censoring,
        gs.week
    FROM base AS b
    CROSS JOIN LATERAL generate_series(0, b.t_final_week_clamped) AS gs(week)
)
SELECT
    *,
    CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation AS enrollment_id,
    CASE
        WHEN event_observed = 1 AND week = t_event_week THEN 1
        ELSE 0
    END AS event_t
FROM expanded
""")

# ------------------------------
# 3) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_person_period_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MAX(week) AS max_week,
    MAX(t_final_week) AS max_t_final_week,
    MAX(t_final_week_clamped) AS max_t_final_week_clamped,
    MIN(t_final_week) AS min_t_final_week,
    MIN(t_final_week_clamped) AS min_t_final_week_clamped,
    SUM(event_t) AS n_total_event_rows,
    SUM(CASE WHEN used_zero_week_fallback_for_censoring = 1 THEN 1 ELSE 0 END) AS n_rows_from_zero_week_fallback,
    COUNT(DISTINCT CASE WHEN used_zero_week_fallback_for_censoring = 1 THEN enrollment_id END) AS n_enrollments_from_zero_week_fallback,
    COUNT(DISTINCT CASE WHEN t_final_week < 0 THEN enrollment_id END) AS n_enrollments_with_negative_t_final_week_raw
FROM person_period_min
""").fetchdf()

# ------------------------------
# 4) Event integrity check
# ------------------------------
event_check = con.execute("""
WITH per_enrollment AS (
    SELECT
        enrollment_id,
        MAX(event_observed) AS event_observed,
        SUM(event_t) AS n_event_rows
    FROM person_period_min
    GROUP BY 1
)
SELECT
    event_observed,
    n_event_rows,
    COUNT(*) AS n_enrollments
FROM per_enrollment
GROUP BY event_observed, n_event_rows
ORDER BY event_observed, n_event_rows
""").fetchdf()

# ------------------------------
# 5) Week distribution
# ------------------------------
week_distribution = con.execute("""
SELECT
    week,
    COUNT(*) AS n_rows,
    SUM(event_t) AS n_events
FROM person_period_min
GROUP BY week
ORDER BY week
""").fetchdf()

# ------------------------------
# 6) Enrollment coverage check
# ------------------------------
coverage_check = con.execute("""
SELECT
    (SELECT COUNT(*) FROM enrollment_survival_ready) AS n_enrollments_survival_ready,
    (SELECT COUNT(DISTINCT enrollment_id) FROM person_period_min) AS n_enrollments_person_period
""").fetchdf()

coverage_check["n_missing_enrollments_after_expansion"] = (
    coverage_check["n_enrollments_survival_ready"] - coverage_check["n_enrollments_person_period"]
)

# ------------------------------
# 7) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "person_period_min.parquet"
csv_path = DATA_OUTPUT_DIR / "person_period_min.csv"
audit_path = TABLES_DIR / "table_person_period_min_audit.csv"
event_check_path = TABLES_DIR / "table_person_period_event_check.csv"
week_dist_path = TABLES_DIR / "table_person_period_week_distribution.csv"
coverage_check_path = TABLES_DIR / "table_person_period_coverage_check.csv"

con.execute(f"COPY person_period_min TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY person_period_min TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
event_check.to_csv(event_check_path, index=False)
week_distribution.to_csv(week_dist_path, index=False)
coverage_check.to_csv(coverage_check_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nPerson-period summary:")
display(audit_summary)

print("\nEnrollment coverage check:")
display(coverage_check)

print("\nEvent integrity check:")
display(event_check)

print("\nWeek distribution (first rows):")
display(week_distribution.head(15))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", event_check_path.resolve())
print("-", week_dist_path.resolve())
print("-", coverage_check_path.resolve())


B1 — Build Minimal Person-Period Skeleton

Person-period summary:


,n_person_period_rows,n_distinct_enrollments,max_week,max_t_final_week,max_t_final_week_clamped,min_t_final_week,min_t_final_week_clamped,n_total_event_rows,n_rows_from_zero_week_fallback,n_enrollments_from_zero_week_fallback,n_enrollments_with_negative_t_final_week_raw
0,775295,32593,63,63,63,0,0,7387.0,3078.0,3078,0



Enrollment coverage check:


,n_enrollments_survival_ready,n_enrollments_person_period,n_missing_enrollments_after_expansion
0,32593,32593,0



Event integrity check:


,event_observed,n_event_rows,n_enrollments
0,0,0.0,25206
1,1,1.0,7387



Week distribution (first rows):


,week,n_rows,n_events
0,0,32593,713.0
1,1,28633,1068.0
2,2,27396,215.0
3,3,26956,360.0
4,4,26363,303.0
5,5,25843,216.0
6,6,25459,246.0
7,7,25010,256.0
8,8,24554,230.0
9,9,24160,220.0



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/person_period_min.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/person_period_min.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_min_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_event_check.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_week_distribution.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_coverage_check.csv


## B2 — Build Weekly VLE Feature Layer


In [4]:
# ==============================================================
# B2 — Build Weekly VLE Feature Layer
# --------------------------------------------------------------
# Purpose:
#   Aggregate studentVle into a weekly LMS activity layer at the
#   enrollment-week level, creating the minimal weekly behavioral
#   signals needed for downstream temporal feature engineering.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: vle_weekly_features
#   - vle_weekly_features.parquet
#   - vle_weekly_features.csv
#   - table_vle_weekly_features_audit.csv
#   - table_vle_weekly_features_week_distribution.csv
# ==============================================================

print("\n" + "=" * 70)
print("B2 — Build Weekly VLE Feature Layer")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUTPUT_DIR / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Build weekly VLE features
# ------------------------------
con.execute("DROP TABLE IF EXISTS vle_weekly_features")

con.execute("""
CREATE TABLE vle_weekly_features AS
SELECT
    id_student,
    code_module,
    code_presentation,
    CAST(FLOOR(date / 7.0) AS INTEGER) AS week,
    SUM(sum_click) AS total_clicks_week,
    COUNT(*) AS n_vle_rows_week,
    1 AS active_this_week,
    COUNT(DISTINCT id_site) AS n_distinct_sites_week
FROM studentVle
GROUP BY
    id_student,
    code_module,
    code_presentation,
    CAST(FLOOR(date / 7.0) AS INTEGER)
""")

# ------------------------------
# 3) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_enrollment_week_rows,
    COUNT(DISTINCT CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation) AS n_distinct_enrollments,
    MIN(week) AS min_week,
    MAX(week) AS max_week,
    SUM(CASE WHEN week < 0 THEN 1 ELSE 0 END) AS n_rows_negative_weeks,
    SUM(CASE WHEN total_clicks_week = 0 THEN 1 ELSE 0 END) AS n_rows_zero_clicks,
    MIN(total_clicks_week) AS min_total_clicks_week,
    MAX(total_clicks_week) AS max_total_clicks_week
FROM vle_weekly_features
""").fetchdf()

# ------------------------------
# 4) Week distribution
# ------------------------------
week_distribution = con.execute("""
SELECT
    week,
    COUNT(*) AS n_rows,
    SUM(total_clicks_week) AS total_clicks_sum,
    AVG(total_clicks_week) AS avg_total_clicks_week
FROM vle_weekly_features
GROUP BY week
ORDER BY week
""").fetchdf()

# ------------------------------
# 5) Coverage relative to survival-ready enrollments
# ------------------------------
coverage_check = con.execute("""
SELECT
    (SELECT COUNT(*) FROM enrollment_survival_ready) AS n_enrollments_survival_ready,
    (SELECT COUNT(DISTINCT CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation)
     FROM vle_weekly_features) AS n_enrollments_with_weekly_vle
""").fetchdf()

coverage_check["n_enrollments_without_weekly_vle"] = (
    coverage_check["n_enrollments_survival_ready"] - coverage_check["n_enrollments_with_weekly_vle"]
)

# ------------------------------
# 6) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "vle_weekly_features.parquet"
csv_path = DATA_OUTPUT_DIR / "vle_weekly_features.csv"
audit_path = TABLES_DIR / "table_vle_weekly_features_audit.csv"
week_dist_path = TABLES_DIR / "table_vle_weekly_features_week_distribution.csv"
coverage_path = TABLES_DIR / "table_vle_weekly_features_coverage_check.csv"

con.execute(f"COPY vle_weekly_features TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY vle_weekly_features TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
week_distribution.to_csv(week_dist_path, index=False)
coverage_check.to_csv(coverage_path, index=False)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nWeekly VLE summary:")
display(audit_summary)

print("\nCoverage relative to enrollment_survival_ready:")
display(coverage_check)

print("\nWeek distribution (first rows):")
display(week_distribution.head(20))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", week_dist_path.resolve())
print("-", coverage_path.resolve())


B2 — Build Weekly VLE Feature Layer

Weekly VLE summary:


,n_enrollment_week_rows,n_distinct_enrollments,min_week,max_week,n_rows_negative_weeks,n_rows_zero_clicks,min_total_clicks_week,max_total_clicks_week
0,627031,29228,-4,38,47593.0,0.0,1.0,6999.0



Coverage relative to enrollment_survival_ready:


,n_enrollments_survival_ready,n_enrollments_with_weekly_vle,n_enrollments_without_weekly_vle
0,32593,29228,3365



Week distribution (first rows):


,week,n_rows,total_clicks_sum,avg_total_clicks_week
0,-4,1076,43298.0,40.239777
1,-3,10532,433407.0,41.151443
2,-2,16176,646001.0,39.935769
3,-1,19809,1025241.0,51.756323
4,0,23658,2013077.0,85.090752
5,1,23168,1797648.0,77.591851
6,2,23822,2307080.0,96.846612
7,3,22135,1663598.0,75.156901
8,4,21661,1505334.0,69.495129
9,5,20361,1228901.0,60.355631



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/vle_weekly_features.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/vle_weekly_features.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_week_distribution.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_coverage_check.csv


## B3 — Enrich Person-Period with Temporal Features


In [5]:
# ==============================================================
# B3 — Enrich Person-Period with Temporal Features
# --------------------------------------------------------------
# Purpose:
#   Enrich the weekly person-period skeleton with minimal LMS
#   behavioral features and derive temporal covariates required
#   for discrete-time survival modeling.
#
# Methodological note:
#   The survival modeling grid in this project starts at week = 0.
#   Although raw LMS activity includes negative weeks (audited in B2),
#   only week >= 0 is used when enriching the modelable person-period
#   table. This keeps the behavioral covariates aligned with the
#   temporal frame used by the discrete-time survival models and avoids
#   mixing pre-period activity with the formal modeling horizon.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: person_period_enriched
#   - person_period_enriched.parquet
#   - person_period_enriched.csv
#   - table_person_period_enriched_audit.csv
#   - table_person_period_enriched_missing_check.csv
# ==============================================================

print("\n" + "=" * 70)
print("B3 — Enrich Person-Period with Temporal Features")
print("=" * 70)
print("Methodological note: only VLE weeks >= 0 were joined into person_period_enriched.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUTPUT_DIR / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Create non-negative weekly VLE layer
# ------------------------------
con.execute("DROP VIEW IF EXISTS vle_weekly_features_nonnegative")

con.execute("""
CREATE VIEW vle_weekly_features_nonnegative AS
SELECT *
FROM vle_weekly_features
WHERE week >= 0
""")

# ------------------------------
# 3) Join person-period skeleton with weekly VLE features
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_enriched_stage1")

con.execute("""
CREATE TABLE person_period_enriched_stage1 AS
SELECT
    ppm.*,

    COALESCE(vwf.total_clicks_week, 0) AS total_clicks_week,
    COALESCE(vwf.n_vle_rows_week, 0) AS n_vle_rows_week,
    COALESCE(vwf.active_this_week, 0) AS active_this_week,
    COALESCE(vwf.n_distinct_sites_week, 0) AS n_distinct_sites_week

FROM person_period_min AS ppm
LEFT JOIN vle_weekly_features_nonnegative AS vwf
    ON ppm.id_student = vwf.id_student
   AND ppm.code_module = vwf.code_module
   AND ppm.code_presentation = vwf.code_presentation
   AND ppm.week = vwf.week
""")

# ------------------------------
# 4) Build enriched person-period table with temporal covariates
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_enriched")

con.execute("""
CREATE TABLE person_period_enriched AS
WITH base AS (
    SELECT
        *,

        SUM(total_clicks_week) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_clicks_until_t,

        SUM(active_this_week) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_active_weeks_until_t,

        MAX(CASE WHEN active_this_week = 1 THEN week ELSE NULL END) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS last_active_week_so_far
    FROM person_period_enriched_stage1
),
with_recency AS (
    SELECT
        *,

        CASE
            WHEN active_this_week = 1 THEN 0
            WHEN last_active_week_so_far IS NOT NULL THEN week - last_active_week_so_far
            ELSE week + 1
        END AS recency
    FROM base
),
with_groups AS (
    SELECT
        *,
        SUM(CASE WHEN active_this_week = 0 THEN 1 ELSE 0 END) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS inactivity_group
    FROM with_recency
)
SELECT
    *,

    CASE
        WHEN active_this_week = 1 THEN
            ROW_NUMBER() OVER (
                PARTITION BY enrollment_id, inactivity_group
                ORDER BY week
            )
        ELSE 0
    END AS streak
FROM with_groups
""")

# ------------------------------
# 5) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_person_period_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MAX(week) AS max_week,
    AVG(total_clicks_week) AS avg_total_clicks_week,
    AVG(cum_clicks_until_t) AS avg_cum_clicks_until_t,
    AVG(active_this_week) AS avg_active_this_week,
    MAX(recency) AS max_recency,
    MAX(streak) AS max_streak
FROM person_period_enriched
""").fetchdf()

# ------------------------------
# 6) Missing check
# ------------------------------
missing_check = con.execute("""
SELECT
    SUM(CASE WHEN total_clicks_week IS NULL THEN 1 ELSE 0 END) AS n_missing_total_clicks_week,
    SUM(CASE WHEN n_vle_rows_week IS NULL THEN 1 ELSE 0 END) AS n_missing_n_vle_rows_week,
    SUM(CASE WHEN active_this_week IS NULL THEN 1 ELSE 0 END) AS n_missing_active_this_week,
    SUM(CASE WHEN n_distinct_sites_week IS NULL THEN 1 ELSE 0 END) AS n_missing_n_distinct_sites_week,
    SUM(CASE WHEN cum_clicks_until_t IS NULL THEN 1 ELSE 0 END) AS n_missing_cum_clicks_until_t,
    SUM(CASE WHEN recency IS NULL THEN 1 ELSE 0 END) AS n_missing_recency,
    SUM(CASE WHEN streak IS NULL THEN 1 ELSE 0 END) AS n_missing_streak
FROM person_period_enriched
""").fetchdf()

# ------------------------------
# 7) Small ordered sample for inspection
# ------------------------------
sample_rows = con.execute("""
SELECT
    enrollment_id,
    week,
    event_t,
    total_clicks_week,
    active_this_week,
    cum_clicks_until_t,
    recency,
    streak
FROM person_period_enriched
ORDER BY enrollment_id, week
LIMIT 30
""").fetchdf()

# ------------------------------
# 8) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "person_period_enriched.parquet"
csv_path = DATA_OUTPUT_DIR / "person_period_enriched.csv"
audit_path = TABLES_DIR / "table_person_period_enriched_audit.csv"
missing_path = TABLES_DIR / "table_person_period_enriched_missing_check.csv"
sample_path = TABLES_DIR / "table_person_period_enriched_sample.csv"

con.execute(f"COPY person_period_enriched TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY person_period_enriched TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
missing_check.to_csv(missing_path, index=False)
sample_rows.to_csv(sample_path, index=False)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nPerson-period enriched summary:")
display(audit_summary)

print("\nMissing check for new temporal features:")
display(missing_check)

print("\nOrdered sample (first rows):")
display(sample_rows)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", sample_path.resolve())


B3 — Enrich Person-Period with Temporal Features
Methodological note: only VLE weeks >= 0 were joined into person_period_enriched.

Person-period enriched summary:


,n_person_period_rows,n_distinct_enrollments,max_week,avg_total_clicks_week,avg_cum_clicks_until_t,avg_active_this_week,max_recency,max_streak
0,775295,32593,63,48.21964,927.730788,0.744715,42,39



Missing check for new temporal features:


,n_missing_total_clicks_week,n_missing_n_vle_rows_week,n_missing_active_this_week,n_missing_n_distinct_sites_week,n_missing_cum_clicks_until_t,n_missing_recency,n_missing_streak
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Ordered sample (first rows):


,enrollment_id,week,event_t,total_clicks_week,active_this_week,cum_clicks_until_t,recency,streak
0,100064|FFF|2013J,0,0,190.0,1,190.0,0,1
1,100064|FFF|2013J,1,0,77.0,1,267.0,0,2
2,100064|FFF|2013J,2,0,17.0,1,284.0,0,3
3,100064|FFF|2013J,3,0,69.0,1,353.0,0,4
4,100064|FFF|2013J,4,0,76.0,1,429.0,0,5
5,100064|FFF|2013J,5,0,135.0,1,564.0,0,6
6,100064|FFF|2013J,6,0,194.0,1,758.0,0,7
7,100064|FFF|2013J,7,0,120.0,1,878.0,0,8
8,100064|FFF|2013J,8,0,156.0,1,1034.0,0,9
9,100064|FFF|2013J,9,0,250.0,1,1284.0,0,10



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/person_period_enriched.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/person_period_enriched.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_enriched_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_enriched_missing_check.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_person_period_enriched_sample.csv


## B4 — Build Enrollment-Level Model Table (Revised v2)


In [6]:
# ==============================================================
# B4 — Build Enrollment-Level Model Table (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Build the canonical enrollment-level model table for the
#   benchmark, containing only structural enrollment-level fields,
#   survival targets, and static covariates.
#
# Methodological note:
#   This revised version intentionally avoids embedding
#   retrospective whole-course temporal summary features (e.g.,
#   all-time activity summaries) into the canonical enrollment-level
#   benchmark table.
#
#   The goal is to preserve a clean structural base that can later
#   be combined with early-window features in a controlled and
#   comparable way, especially for enrollment-level models such as
#   Cox.
#
# Inputs expected from previous cells:
#   - con
#   - DATA_OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - enrollment_model_table.parquet
#   - enrollment_model_table.csv
#   - table_enrollment_model_table_audit.csv
#   - table_enrollment_model_table_missing_check.csv
#   - table_enrollment_model_table_event_distribution.csv
#   - table_enrollment_model_table_sample.csv
# ==============================================================

print("\n" + "=" * 70)
print("B4 — Build Enrollment-Level Model Table (Revised v2)")
print("=" * 70)
print("Methodological note: this revised table is a canonical structural")
print("enrollment-level base and intentionally excludes all-time temporal")
print("summary features from the benchmark core.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "DATA_OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
import numpy as np

# ------------------------------
# 2) Build revised canonical table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_model_table")

con.execute("""
CREATE TABLE enrollment_model_table AS
SELECT
    CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation AS enrollment_id,
    id_student,
    code_module,
    code_presentation,

    -- survival targets
    event_observed AS event,
    t_last_obs_week AS duration_raw,
    CASE
        WHEN t_last_obs_week < 0 THEN 0
        ELSE t_last_obs_week
    END AS duration,

    -- structural note
    used_zero_week_fallback_for_censoring,

    -- static covariates
    gender,
    region,
    highest_education,
    imd_band,
    age_band,
    num_of_prev_attempts,
    studied_credits,
    disability

FROM enrollment_survival_ready
""")

# ------------------------------
# 3) Load for audit/export
# ------------------------------
enrollment_model_table = con.execute("""
SELECT *
FROM enrollment_model_table
ORDER BY enrollment_id
""").fetchdf()

# ------------------------------
# 4) Audit summaries
# ------------------------------
summary_df = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MIN(duration_raw) AS min_duration_raw,
    MIN(duration) AS min_duration,
    AVG(duration) AS avg_duration,
    MAX(duration) AS max_duration,
    SUM(event) AS n_events,
    100.0 * AVG(event) AS pct_events,
    SUM(CASE WHEN duration_raw < 0 THEN 1 ELSE 0 END) AS n_negative_duration_raw,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT enrollment_id) THEN TRUE
        ELSE FALSE
    END AS is_unique_by_enrollment_id
FROM enrollment_model_table
""").fetchdf()

event_distribution_df = con.execute("""
SELECT
    event,
    COUNT(*) AS n
FROM enrollment_model_table
GROUP BY event
ORDER BY event
""").fetchdf()

missing_check_df = con.execute("""
SELECT
    SUM(CASE WHEN duration_raw IS NULL THEN 1 ELSE 0 END) AS n_missing_duration_raw,
    SUM(CASE WHEN duration IS NULL THEN 1 ELSE 0 END) AS n_missing_duration,
    SUM(CASE WHEN event IS NULL THEN 1 ELSE 0 END) AS n_missing_event,
    SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS n_missing_gender,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS n_missing_region,
    SUM(CASE WHEN highest_education IS NULL THEN 1 ELSE 0 END) AS n_missing_highest_education,
    SUM(CASE WHEN imd_band IS NULL THEN 1 ELSE 0 END) AS n_missing_imd_band,
    SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END) AS n_missing_age_band,
    SUM(CASE WHEN num_of_prev_attempts IS NULL THEN 1 ELSE 0 END) AS n_missing_num_of_prev_attempts,
    SUM(CASE WHEN studied_credits IS NULL THEN 1 ELSE 0 END) AS n_missing_studied_credits,
    SUM(CASE WHEN disability IS NULL THEN 1 ELSE 0 END) AS n_missing_disability,
    SUM(CASE WHEN used_zero_week_fallback_for_censoring IS NULL THEN 1 ELSE 0 END) AS n_missing_zero_week_fallback_flag
FROM enrollment_model_table
""").fetchdf()

sample_df = con.execute("""
SELECT *
FROM enrollment_model_table
ORDER BY enrollment_id
LIMIT 30
""").fetchdf()

# ------------------------------
# 5) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_model_table.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_model_table.csv"
audit_path = TABLES_DIR / "table_enrollment_model_table_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_model_table_missing_check.csv"
event_dist_path = TABLES_DIR / "table_enrollment_model_table_event_distribution.csv"
sample_path = TABLES_DIR / "table_enrollment_model_table_sample.csv"

enrollment_model_table.to_parquet(parquet_path, index=False)
enrollment_model_table.to_csv(csv_path, index=False)

summary_df.to_csv(audit_path, index=False)
missing_check_df.to_csv(missing_path, index=False)
event_distribution_df.to_csv(event_dist_path, index=False)
sample_df.to_csv(sample_path, index=False)

# ------------------------------
# 6) Output for feedback
# ------------------------------
print("\nEnrollment model table summary:")
display(summary_df)

print("\nEvent distribution:")
display(event_distribution_df)

print("\nMissing check:")
display(missing_check_df)

print("\nSample rows:")
display(sample_df)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", event_dist_path.resolve())
print("-", sample_path.resolve())


B4 — Build Enrollment-Level Model Table (Revised v2)
Methodological note: this revised table is a canonical structural
enrollment-level base and intentionally excludes all-time temporal
summary features from the benchmark core.

Enrollment model table summary:


,n_rows,n_distinct_enrollments,min_duration_raw,min_duration,avg_duration,max_duration,n_events,pct_events,n_negative_duration_raw,is_unique_by_enrollment_id
0,32593,32593,0,0,22.35256,38,7387.0,22.664376,0.0,True



Event distribution:


,event,n
0,0,25206
1,1,7387



Missing check:


,n_missing_duration_raw,n_missing_duration,n_missing_event,n_missing_gender,n_missing_region,n_missing_highest_education,n_missing_imd_band,n_missing_age_band,n_missing_num_of_prev_attempts,n_missing_studied_credits,n_missing_disability,n_missing_zero_week_fallback_flag
0,0.0,0.0,0.0,0.0,0.0,0.0,1111.0,0.0,0.0,0.0,0.0,0.0



Sample rows:


,enrollment_id,id_student,code_module,code_presentation,event,duration_raw,duration,used_zero_week_fallback_for_censoring,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability
0,100064|FFF|2013J,100064,FFF,2013J,0,38,38,0,F,West Midlands Region,A Level or Equivalent,0-10%,35-55,0,60,N
1,100282|BBB|2013J,100282,BBB,2013J,1,5,5,0,F,Wales,Lower Than A Level,20-30%,0-35,1,120,N
2,100561|DDD|2014J,100561,DDD,2014J,0,34,34,0,M,East Anglian Region,Lower Than A Level,70-80%,0-35,1,60,N
3,100621|CCC|2014B,100621,CCC,2014B,1,3,3,0,M,West Midlands Region,A Level or Equivalent,50-60%,35-55,0,60,N
4,1006742|FFF|2014B,1006742,FFF,2014B,0,14,14,0,M,Scotland,HE Qualification,80-90%,55<=,1,120,N
5,100758|EEE|2013J,100758,EEE,2013J,0,0,0,1,F,London Region,A Level or Equivalent,10-20,0-35,0,150,N
6,100788|CCC|2014J,100788,CCC,2014J,0,34,34,0,M,Scotland,HE Qualification,80-90%,0-35,1,60,N
7,100788|FFF|2013J,100788,FFF,2013J,0,38,38,0,M,Scotland,HE Qualification,80-90%,0-35,0,90,N
8,1008675|BBB|2013B,1008675,BBB,2013B,0,34,34,0,F,East Anglian Region,Lower Than A Level,90-100%,35-55,0,60,N
9,100893|AAA|2013J,100893,AAA,2013J,0,35,35,0,M,Yorkshire Region,A Level or Equivalent,20-30%,0-35,0,60,N



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_model_table.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_model_table.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_model_table_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_model_table_missing_check.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_model_table_event_distribution.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_model_table_sample.csv


## B5 — Define Canonical Modeling Configuration Layer


In [7]:
# ==============================================================



# B5 — Define Canonical Modeling Configuration Layer



# 



# ==============================================================



# Purpose:



#   Centralize the benchmark configuration under the corrected



#   benchmark architecture.



#



# This step:



#   - preserves current operational configuration objects



#   - adds explicit benchmark-arm structure



#   - officially incorporates the future



#     discrete_time_comparable_minimal arm



#   - keeps downstream compatibility as much as possible



#



# Main outputs:



#   - benchmark configuration objects in memory



#   - table_p10_canonical_modeling_configuration.csv



#   - metadata_p10_canonical_modeling_configuration.json



# ==============================================================







import json



from pathlib import Path







import pandas as pd







print("=" * 70)



print("B5 — Define Canonical Modeling Configuration Layer")



print("=" * 70)







OUT_BASE = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path(globals().get("SHARED_CONFIG", {}).get("paths", {}).get("output_dir", "outputs_benchmark_survival"))

OUT_TABLES = TABLES_DIR if "TABLES_DIR" in globals() else OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("tables_subdir", "tables")

OUT_METADATA = METADATA_DIR if "METADATA_DIR" in globals() else OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("metadata_subdir", "metadata")

OUT_TABLES.mkdir(parents=True, exist_ok=True)

OUT_METADATA.mkdir(parents=True, exist_ok=True)







# --------------------------------------------------------------



# 1) Core benchmark constants



# --------------------------------------------------------------



_shared_cfg = globals().get("SHARED_BENCHMARK_CONFIG", {})
RANDOM_SEED = int(_shared_cfg.get("seed", 42))
TEST_SIZE = float(_shared_cfg.get("test_size", 0.30))
EARLY_WINDOW_WEEKS = int(_shared_cfg.get("early_window_weeks", 4))
MAIN_ENROLLMENT_WINDOW_WEEKS = int(_shared_cfg.get("main_enrollment_window_weeks", EARLY_WINDOW_WEEKS))

MAIN_CLICKS_FEATURE = f"clicks_first_{MAIN_ENROLLMENT_WINDOW_WEEKS}_weeks"
MAIN_ACTIVE_FEATURE = f"active_weeks_first_{MAIN_ENROLLMENT_WINDOW_WEEKS}"
MAIN_MEAN_CLICKS_FEATURE = f"mean_clicks_first_{MAIN_ENROLLMENT_WINDOW_WEEKS}_weeks"







# --------------------------------------------------------------



# 2) Feature groups



# --------------------------------------------------------------



# Keep names aligned with the current notebook convention.







STATIC_FEATURES = [



    "studied_credits",



    "num_of_prev_attempts",



]







TEMPORAL_FEATURES_DISCRETE = [



    "week",



    "total_clicks",



]







# Canonical enrollment-level early-window feature set



MAIN_ENROLLMENT_WINDOW_FEATURES = [



    "studied_credits",



    "num_of_prev_attempts",



    MAIN_CLICKS_FEATURE,



]







COMPARABLE_ARM_FEATURES = MAIN_ENROLLMENT_WINDOW_FEATURES.copy()







DYNAMIC_ARM_FEATURES_LINEAR = [



    "week",



    "total_clicks",



    "studied_credits",



    "num_of_prev_attempts",



]







DYNAMIC_ARM_FEATURES_NEURAL = [



    "week",



    "total_clicks",



    "studied_credits",



    "num_of_prev_attempts",



]







# Optional extra comparable-window features that may exist in some



# upstream tables but are not required in the minimal canonical set.



OPTIONAL_COMPARABLE_WINDOW_FEATURES = [



    MAIN_ACTIVE_FEATURE,



    MAIN_MEAN_CLICKS_FEATURE,



]







# --------------------------------------------------------------



# 3) Canonical benchmark architecture config



# --------------------------------------------------------------



BENCHMARK_ARMS = {



    "comparable_arm": {



        "description": "Enrollment-level / static-after-early-window comparable benchmark arm.",



        "representation_type": "enrollment_level_or_early_window_summary",



        "update_regime": "static_after_early_window",



        "families": [



            "continuous_time",



            "continuous_time_neural",



            "discrete_time_comparable_minimal",



        ],



    },



    "dynamic_arm": {



        "description": "Weekly-updating person-period dynamic benchmark arm.",



        "representation_type": "dynamic_weekly_person_period",



        "update_regime": "weekly_updating",



        "families": [



            "discrete_time",



            "discrete_time_neural",



        ],



    },



}







EVALUATION_OVERLAYS = {



    "main_split": {



        "description": "Main benchmark split used by the original benchmark.",



        "table_or_protocol": "existing train/test ready tables",



        "currently_active": True,



    },



    "context_heldout_v1": {



        "description": "Alternative grouped context-held-out split for transportability analysis.",



        "table_or_protocol": "enrollment_survival_ready_context_split",



        "currently_active": False,



    },



}







# --------------------------------------------------------------



# 4) Canonical model input variants



# --------------------------------------------------------------



MODEL_INPUT_VARIANTS = {



    "continuous_time": {



        "benchmark_arm": "comparable_arm",



        "family": "continuous_time",



        "representation_type": "enrollment_level_or_early_window_summary",



        "update_regime": "static_after_early_window",



        "input_table_train": "enrollment_cox_ready_train",



        "input_table_test": "enrollment_cox_ready_test",



        "feature_set": COMPARABLE_ARM_FEATURES,



        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,



        "currently_materialized": True,



        "included_in_main_results": True,



    },



    "continuous_time_neural": {



        "benchmark_arm": "comparable_arm",



        "family": "continuous_time_neural",



        "representation_type": "enrollment_level_or_early_window_summary",



        "update_regime": "static_after_early_window",



        "input_table_train": "enrollment_deepsurv_ready_train",



        "input_table_test": "enrollment_deepsurv_ready_test",



        "feature_set": COMPARABLE_ARM_FEATURES,



        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,



        "currently_materialized": True,



        "included_in_main_results": True,



    },



    "discrete_time_comparable_minimal": {



        "benchmark_arm": "comparable_arm",



        "family": "discrete_time_comparable_minimal",



        "representation_type": "enrollment_level_or_early_window_summary",



        "update_regime": "static_after_early_window",



        "input_table_train": "to_be_materialized",



        "input_table_test": "to_be_materialized",



        "feature_set": COMPARABLE_ARM_FEATURES,



        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,



        "currently_materialized": False,



        "included_in_main_results": False,



    },



    "discrete_time": {



        "benchmark_arm": "dynamic_arm",



        "family": "discrete_time",



        "representation_type": "dynamic_weekly_person_period",



        "update_regime": "weekly_updating",



        "input_table_train": "pp_linear_hazard_ready_train",



        "input_table_test": "pp_linear_hazard_ready_test",



        "feature_set": DYNAMIC_ARM_FEATURES_LINEAR,



        "optional_feature_set": [],



        "currently_materialized": True,



        "included_in_main_results": True,



    },



    "discrete_time_neural": {



        "benchmark_arm": "dynamic_arm",



        "family": "discrete_time_neural",



        "representation_type": "dynamic_weekly_person_period",



        "update_regime": "weekly_updating",



        "input_table_train": "pp_neural_hazard_ready_train",



        "input_table_test": "pp_neural_hazard_ready_test",



        "feature_set": DYNAMIC_ARM_FEATURES_NEURAL,



        "optional_feature_set": [],



        "currently_materialized": True,



        "included_in_main_results": True,



    },



}







# --------------------------------------------------------------



# 5) Backward-compatible high-level config object



# --------------------------------------------------------------



MODELING_CONFIG = {



    "random_seed": RANDOM_SEED,



    "test_size": TEST_SIZE,



    "early_window_weeks": EARLY_WINDOW_WEEKS,



    "main_enrollment_window_weeks": MAIN_ENROLLMENT_WINDOW_WEEKS,



    "static_features": STATIC_FEATURES,



    "temporal_features_discrete": TEMPORAL_FEATURES_DISCRETE,



    "main_enrollment_window_features": MAIN_ENROLLMENT_WINDOW_FEATURES,



    "optional_comparable_window_features": OPTIONAL_COMPARABLE_WINDOW_FEATURES,



    "benchmark_arms": BENCHMARK_ARMS,



    "evaluation_overlays": EVALUATION_OVERLAYS,



    "model_input_variants": MODEL_INPUT_VARIANTS,



    "main_interpretation_note": (



        "Family-level ranking must be interpreted within benchmark arm. "



        "Cross-arm comparisons are informative but not pure architecture-only comparisons."



    ),



}







# --------------------------------------------------------------



# 6) Tabular export for auditability



# --------------------------------------------------------------



rows = []



for variant_name, cfg in MODEL_INPUT_VARIANTS.items():



    rows.append({



        "variant_name": variant_name,



        "benchmark_arm": cfg["benchmark_arm"],



        "family": cfg["family"],



        "representation_type": cfg["representation_type"],



        "update_regime": cfg["update_regime"],



        "input_table_train": cfg["input_table_train"],



        "input_table_test": cfg["input_table_test"],



        "feature_set": " | ".join(cfg["feature_set"]),



        "optional_feature_set": " | ".join(cfg["optional_feature_set"]),



        "currently_materialized": cfg["currently_materialized"],



        "included_in_main_results": cfg["included_in_main_results"],



    })







table_p10_canonical_modeling_configuration = pd.DataFrame(rows).sort_values(



    ["benchmark_arm", "variant_name"]



).reset_index(drop=True)







path_table = OUT_TABLES / "table_p10_canonical_modeling_configuration.csv"



path_metadata = OUT_METADATA / "metadata_p10_canonical_modeling_configuration.json"







table_p10_canonical_modeling_configuration.to_csv(path_table, index=False)







metadata = {



    "step": "B5",



    "title": "Define Canonical Modeling Configuration Layer",



    "random_seed": RANDOM_SEED,



    "test_size": TEST_SIZE,



    "early_window_weeks": EARLY_WINDOW_WEEKS,



    "main_enrollment_window_weeks": MAIN_ENROLLMENT_WINDOW_WEEKS,



    "benchmark_arms": list(BENCHMARK_ARMS.keys()),



    "evaluation_overlays": list(EVALUATION_OVERLAYS.keys()),



    "model_variants": list(MODEL_INPUT_VARIANTS.keys()),



    "canonical_comparable_click_feature": MAIN_CLICKS_FEATURE,



    "notes": [



        "B5 now follows the corrected benchmark architecture.",



        "discrete_time_comparable_minimal is now officially part of the canonical configuration.",



        f"The canonical comparable click feature is {MAIN_CLICKS_FEATURE}, aligned with the notebook.",



        "This step defines configuration only; materialization of the new comparable discrete-time arm occurs downstream."



    ],



    "output_tables": [



        path_table.as_posix(),



    ],



}



with open(path_metadata, "w", encoding="utf-8") as f:



    json.dump(metadata, f, indent=2)







# --------------------------------------------------------------



# 7) Display



# --------------------------------------------------------------



print("\nCanonical model input variants:")



display(table_p10_canonical_modeling_configuration)







print("\nKey config objects created:")



print("- RANDOM_SEED")



print("- TEST_SIZE")



print("- EARLY_WINDOW_WEEKS")



print("- MAIN_ENROLLMENT_WINDOW_WEEKS")



print("- STATIC_FEATURES")



print("- TEMPORAL_FEATURES_DISCRETE")



print("- MAIN_ENROLLMENT_WINDOW_FEATURES")



print("- OPTIONAL_COMPARABLE_WINDOW_FEATURES")



print("- BENCHMARK_ARMS")



print("- EVALUATION_OVERLAYS")



print("- MODEL_INPUT_VARIANTS")



print("- MODELING_CONFIG")







print("\nSaved:")



print("-", path_table.resolve())



print("-", path_metadata.resolve())

B5 — Define Canonical Modeling Configuration Layer

Canonical model input variants:


,variant_name,benchmark_arm,family,representation_type,update_regime,input_table_train,input_table_test,feature_set,optional_feature_set,currently_materialized,included_in_main_results
0,continuous_time,comparable_arm,continuous_time,enrollment_level_or_early_window_summary,static_after_early_window,enrollment_cox_ready_train,enrollment_cox_ready_test,studied_credits | num_of_prev_attempts | click...,active_weeks_first_4 | mean_clicks_first_4_weeks,True,True
1,continuous_time_neural,comparable_arm,continuous_time_neural,enrollment_level_or_early_window_summary,static_after_early_window,enrollment_deepsurv_ready_train,enrollment_deepsurv_ready_test,studied_credits | num_of_prev_attempts | click...,active_weeks_first_4 | mean_clicks_first_4_weeks,True,True
2,discrete_time_comparable_minimal,comparable_arm,discrete_time_comparable_minimal,enrollment_level_or_early_window_summary,static_after_early_window,to_be_materialized,to_be_materialized,studied_credits | num_of_prev_attempts | click...,active_weeks_first_4 | mean_clicks_first_4_weeks,False,False
3,discrete_time,dynamic_arm,discrete_time,dynamic_weekly_person_period,weekly_updating,pp_linear_hazard_ready_train,pp_linear_hazard_ready_test,week | total_clicks | studied_credits | num_of...,,True,True
4,discrete_time_neural,dynamic_arm,discrete_time_neural,dynamic_weekly_person_period,weekly_updating,pp_neural_hazard_ready_train,pp_neural_hazard_ready_test,week | total_clicks | studied_credits | num_of...,,True,True



Key config objects created:
- RANDOM_SEED
- TEST_SIZE
- EARLY_WINDOW_WEEKS
- MAIN_ENROLLMENT_WINDOW_WEEKS
- STATIC_FEATURES
- TEMPORAL_FEATURES_DISCRETE
- MAIN_ENROLLMENT_WINDOW_FEATURES
- OPTIONAL_COMPARABLE_WINDOW_FEATURES
- BENCHMARK_ARMS
- EVALUATION_OVERLAYS
- MODEL_INPUT_VARIANTS
- MODELING_CONFIG

Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_p10_canonical_modeling_configuration.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/metadata_p10_canonical_modeling_configuration.json


### B5.1 — Define Corrected Benchmark Architecture


### Funcao: add_row

Definicao isolada de add_row para reutilizacao nas etapas seguintes.


In [8]:
# ============================================================
# B5.1 — Define Corrected Benchmark Architecture
# ============================================================
# Purpose:
#   Formalize the corrected benchmark architecture after the main configuration layer.
#   This step does NOT retrain models. It defines:
#     - benchmark arms
#     - family membership by arm
#     - expected input tables
#     - comparability scope
#     - evaluation overlays
#
# Main outputs:
#   - table_p10_1_benchmark_architecture.csv
#   - table_p10_1_benchmark_arm_summary.csv
#   - table_p10_1_benchmark_overlay_summary.csv
#   - metadata_p10_1_benchmark_architecture.json
# ============================================================

import json
from pathlib import Path

import pandas as pd

print("=" * 70)
print("B5.1 — Define Corrected Benchmark Architecture")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path(globals().get("SHARED_CONFIG", {}).get("paths", {}).get("output_dir", "outputs_benchmark_survival"))
OUT_TABLES = TABLES_DIR if "TABLES_DIR" in globals() else OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("tables_subdir", "tables")
OUT_METADATA = METADATA_DIR if "METADATA_DIR" in globals() else OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("metadata_subdir", "metadata")
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = sorted(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------------------------------------
# 1) Build official architecture table
# ------------------------------------------------------------
rows = []

def add_row(
    benchmark_arm,
    arm_role,
    family,
    model_label,
    status_in_project,
    current_or_future,
    representation_type,
    update_regime,
    expected_input_table,
    expected_split_protocol,
    comparability_scope,
    included_in_current_main_results,
    notes
):
    rows.append({
        "benchmark_arm": benchmark_arm,
        "arm_role": arm_role,
        "family": family,
        "model_label": model_label,
        "status_in_project": status_in_project,
        "current_or_future": current_or_future,
        "representation_type": representation_type,
        "update_regime": update_regime,
        "expected_input_table": expected_input_table,
        "expected_split_protocol": expected_split_protocol,
        "comparability_scope": comparability_scope,
        "included_in_current_main_results": included_in_current_main_results,
        "notes": notes,
    })

# ------------------------------------------------------------
# 1A) Comparable benchmark arm
# ------------------------------------------------------------
add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="continuous_time",
    model_label="Comparable Cox arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="enrollment_cox_*",
    expected_split_protocol="main_split and future context-held-out reruns",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=True,
    notes="Continuous-time comparable arm already exists and operates on static/enrollment-level inputs."
)

add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="continuous_time_neural",
    model_label="Comparable DeepSurv arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="enrollment_deepsurv_*",
    expected_split_protocol="main_split and future context-held-out reruns",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=True,
    notes="Neural continuous-time comparable arm already exists and operates on static/enrollment-level inputs."
)

add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="discrete_time_comparable_minimal",
    model_label="Comparable discrete-time minimal arm",
    status_in_project="to_be_materialized",
    current_or_future="future",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="to_be_defined_from_enrollment_survival_ready / enrollment-level comparable features",
    expected_split_protocol="main_split first, context-held-out optionally later",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=False,
    notes="This future arm is the minimum structural correction needed to compare discrete-time models under the same representation regime as Cox/DeepSurv comparable arms."
)

# ------------------------------------------------------------
# 1B) Dynamic benchmark arm
# ------------------------------------------------------------
add_row(
    benchmark_arm="dynamic_arm",
    arm_role="model_family",
    family="discrete_time",
    model_label="Dynamic linear hazard arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="weekly_updating",
    expected_input_table="pp_linear_hazard_*",
    expected_split_protocol="main_split and future context-held-out reruns if materialized",
    comparability_scope="within dynamic_arm",
    included_in_current_main_results=True,
    notes="Dynamic discrete-time linear hazard arm already exists and uses person-period rows."
)

add_row(
    benchmark_arm="dynamic_arm",
    arm_role="model_family",
    family="discrete_time_neural",
    model_label="Dynamic neural hazard arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="weekly_updating",
    expected_input_table="pp_neural_hazard_*",
    expected_split_protocol="main_split and future context-held-out reruns if materialized",
    comparability_scope="within dynamic_arm",
    included_in_current_main_results=True,
    notes="Dynamic discrete-time neural hazard arm already exists and uses person-period rows."
)

# ------------------------------------------------------------
# 1C) Upstream backbones
# ------------------------------------------------------------
add_row(
    benchmark_arm="shared_backbone",
    arm_role="upstream_data",
    family="enrollment_survival_backbone",
    model_label="Enrollment survival backbone",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_survival",
    update_regime="not_a_model",
    expected_input_table="enrollment_survival_ready",
    expected_split_protocol="feeds all benchmark arms",
    comparability_scope="shared_input_backbone",
    included_in_current_main_results=False,
    notes="Canonical event/time enrollment-level backbone."
)

add_row(
    benchmark_arm="shared_backbone",
    arm_role="upstream_data",
    family="person_period_backbone",
    model_label="Person-period dynamic backbone",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="not_a_model",
    expected_input_table="person_period_* / pp_*",
    expected_split_protocol="feeds dynamic benchmark arm",
    comparability_scope="shared_input_backbone",
    included_in_current_main_results=False,
    notes="Canonical dynamic weekly backbone for person-period models."
)

# ------------------------------------------------------------
# 1D) Evaluation overlays
# ------------------------------------------------------------
add_row(
    benchmark_arm="evaluation_overlay",
    arm_role="evaluation_protocol",
    family="main_random_enrollment_split",
    model_label="Main benchmark split",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="split_protocol",
    update_regime="not_a_model",
    expected_input_table="existing train/test ready tables",
    expected_split_protocol="main_split",
    comparability_scope="applies_across_arms",
    included_in_current_main_results=True,
    notes="Main benchmark protocol currently used in the notebook and paper."
)

if "enrollment_survival_ready_context_split" in available_tables:
    add_row(
        benchmark_arm="evaluation_overlay",
        arm_role="evaluation_protocol",
        family="context_heldout_split",
        model_label="Context-held-out split",
        status_in_project="already_present",
        current_or_future="current",
        representation_type="split_protocol",
        update_regime="not_a_model",
        expected_input_table="enrollment_survival_ready_context_split",
        expected_split_protocol="context_heldout_v1",
        comparability_scope="applies_across_arms",
        included_in_current_main_results=False,
        notes="Alternative transportability-oriented split protocol holding out entire (module, presentation) groups."
    )

table_p10_1_benchmark_architecture = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2) Arm-level summary
# ------------------------------------------------------------
table_p10_1_benchmark_arm_summary = (
    table_p10_1_benchmark_architecture
    .groupby(
        ["benchmark_arm", "representation_type", "update_regime", "comparability_scope"],
        dropna=False
    )
    .size()
    .reset_index(name="n_entries")
    .sort_values(["benchmark_arm", "representation_type", "update_regime"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3) Overlay summary
# ------------------------------------------------------------
table_p10_1_benchmark_overlay_summary = (
    table_p10_1_benchmark_architecture
    .loc[table_p10_1_benchmark_architecture["arm_role"] == "evaluation_protocol"]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4) Save outputs
# ------------------------------------------------------------
path_arch = OUT_TABLES / "table_p10_1_benchmark_architecture.csv"
path_arm_summary = OUT_TABLES / "table_p10_1_benchmark_arm_summary.csv"
path_overlay_summary = OUT_TABLES / "table_p10_1_benchmark_overlay_summary.csv"
path_metadata = OUT_METADATA / "metadata_p10_1_benchmark_architecture.json"

table_p10_1_benchmark_architecture.to_csv(path_arch, index=False)
table_p10_1_benchmark_arm_summary.to_csv(path_arm_summary, index=False)
table_p10_1_benchmark_overlay_summary.to_csv(path_overlay_summary, index=False)

metadata = {
    "step": "B5.1",
    "title": "Define Corrected Benchmark Architecture",
    "main_decision": (
        "Benchmark is formally separated into a comparable arm and a dynamic arm, "
        "with evaluation overlays treated as protocols rather than model families."
    ),
    "comparable_arm_definition": (
        "Enrollment-level or early-window summary representations; "
        "static-after-early-window updating regime."
    ),
    "dynamic_arm_definition": (
        "Dynamic weekly person-period representations; weekly-updating regime."
    ),
    "future_structural_task": (
        "Materialize a minimal comparable discrete-time arm so that discrete-time "
        "models can also be compared under the same representation regime as "
        "the comparable Cox and DeepSurv arms."
    ),
    "output_tables": [
        path_arch.as_posix(),
        path_arm_summary.as_posix(),
        path_overlay_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 5) Display
# ------------------------------------------------------------
print("\nBenchmark architecture:")
display(table_p10_1_benchmark_architecture)

print("\nBenchmark arm summary:")
display(table_p10_1_benchmark_arm_summary)

print("\nBenchmark overlay summary:")
display(table_p10_1_benchmark_overlay_summary)

print("\nSaved:")
print("-", path_arch.resolve())
print("-", path_arm_summary.resolve())
print("-", path_overlay_summary.resolve())
print("-", path_metadata.resolve())

B5.1 — Define Corrected Benchmark Architecture

Benchmark architecture:


,benchmark_arm,arm_role,family,model_label,status_in_project,current_or_future,representation_type,update_regime,expected_input_table,expected_split_protocol,comparability_scope,included_in_current_main_results,notes
0,comparable_arm,model_family,continuous_time,Comparable Cox arm,already_present,current,enrollment_level_or_early_window_summary,static_after_early_window,enrollment_cox_*,main_split and future context-held-out reruns,within comparable_arm,True,Continuous-time comparable arm already exists ...
1,comparable_arm,model_family,continuous_time_neural,Comparable DeepSurv arm,already_present,current,enrollment_level_or_early_window_summary,static_after_early_window,enrollment_deepsurv_*,main_split and future context-held-out reruns,within comparable_arm,True,Neural continuous-time comparable arm already ...
2,comparable_arm,model_family,discrete_time_comparable_minimal,Comparable discrete-time minimal arm,to_be_materialized,future,enrollment_level_or_early_window_summary,static_after_early_window,to_be_defined_from_enrollment_survival_ready /...,"main_split first, context-held-out optionally ...",within comparable_arm,False,This future arm is the minimum structural corr...
3,dynamic_arm,model_family,discrete_time,Dynamic linear hazard arm,already_present,current,dynamic_weekly_person_period,weekly_updating,pp_linear_hazard_*,main_split and future context-held-out reruns ...,within dynamic_arm,True,Dynamic discrete-time linear hazard arm alread...
4,dynamic_arm,model_family,discrete_time_neural,Dynamic neural hazard arm,already_present,current,dynamic_weekly_person_period,weekly_updating,pp_neural_hazard_*,main_split and future context-held-out reruns ...,within dynamic_arm,True,Dynamic discrete-time neural hazard arm alread...
5,shared_backbone,upstream_data,enrollment_survival_backbone,Enrollment survival backbone,already_present,current,enrollment_level_survival,not_a_model,enrollment_survival_ready,feeds all benchmark arms,shared_input_backbone,False,Canonical event/time enrollment-level backbone.
6,shared_backbone,upstream_data,person_period_backbone,Person-period dynamic backbone,already_present,current,dynamic_weekly_person_period,not_a_model,person_period_* / pp_*,feeds dynamic benchmark arm,shared_input_backbone,False,Canonical dynamic weekly backbone for person-p...
7,evaluation_overlay,evaluation_protocol,main_random_enrollment_split,Main benchmark split,already_present,current,split_protocol,not_a_model,existing train/test ready tables,main_split,applies_across_arms,True,Main benchmark protocol currently used in the ...



Benchmark arm summary:


,benchmark_arm,representation_type,update_regime,comparability_scope,n_entries
0,comparable_arm,enrollment_level_or_early_window_summary,static_after_early_window,within comparable_arm,3
1,dynamic_arm,dynamic_weekly_person_period,weekly_updating,within dynamic_arm,2
2,evaluation_overlay,split_protocol,not_a_model,applies_across_arms,1
3,shared_backbone,dynamic_weekly_person_period,not_a_model,shared_input_backbone,1
4,shared_backbone,enrollment_level_survival,not_a_model,shared_input_backbone,1



Benchmark overlay summary:


,benchmark_arm,arm_role,family,model_label,status_in_project,current_or_future,representation_type,update_regime,expected_input_table,expected_split_protocol,comparability_scope,included_in_current_main_results,notes
0,evaluation_overlay,evaluation_protocol,main_random_enrollment_split,Main benchmark split,already_present,current,split_protocol,not_a_model,existing train/test ready tables,main_split,applies_across_arms,True,Main benchmark protocol currently used in the ...



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_p10_1_benchmark_architecture.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_p10_1_benchmark_arm_summary.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_p10_1_benchmark_overlay_summary.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/metadata/metadata_p10_1_benchmark_architecture.json


### B5.2 — Materialize Canonical Enrollment Window Features

In [ ]:
# ==============================================================
# B5.2 — Materialize Canonical Enrollment Window Features
# --------------------------------------------------------------
# Purpose:
#   Materialize the canonical enrollment-level early-window
#   features used by the comparable enrollment-level benchmark arms.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - MAIN_ENROLLMENT_WINDOW_WEEKS
#   - person_period_enriched
#
# Outputs:
#   - DuckDB table: enrollment_window_features
#   - enrollment_window_features.parquet
#   - enrollment_window_features.csv
#   - table_enrollment_window_features_audit.csv
#   - table_enrollment_window_features_missing_check.csv
# ==============================================================

print("\n" + "=" * 70)
print("B5.2 — Materialize Canonical Enrollment Window Features")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR", "MAIN_ENROLLMENT_WINDOW_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

required_tables = ["person_period_enriched"]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required DuckDB table(s)/view(s): "
        + ", ".join(missing_tables)
        + ". Please run the required upstream cells first."
    )

DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUTPUT_DIR / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

window_weeks = int(MAIN_ENROLLMENT_WINDOW_WEEKS)

# ------------------------------
# 2) Materialize canonical table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_window_features")

con.execute(f"""
CREATE TABLE enrollment_window_features AS
WITH early_window AS (
    SELECT
        enrollment_id,
        week,
        total_clicks_week,
        active_this_week,
        n_distinct_sites_week
    FROM person_period_enriched
    WHERE week >= 0
      AND week < {window_weeks}
),
agg AS (
    SELECT
        enrollment_id,
        SUM(COALESCE(total_clicks_week, 0)) AS clicks_first_{window_weeks}_weeks,
        SUM(COALESCE(active_this_week, 0)) AS active_weeks_first_{window_weeks},
        AVG(COALESCE(total_clicks_week, 0)) AS mean_clicks_first_{window_weeks}_weeks,
        AVG(COALESCE(n_distinct_sites_week, 0)) AS mean_distinct_sites_first_{window_weeks}_weeks,
        MAX(week) AS max_week_observed_in_window
    FROM early_window
    GROUP BY enrollment_id
)
SELECT
    enrollment_id,
    clicks_first_{window_weeks}_weeks,
    active_weeks_first_{window_weeks},
    mean_clicks_first_{window_weeks}_weeks,
    mean_distinct_sites_first_{window_weeks}_weeks,
    max_week_observed_in_window
FROM agg
""")

window_df = con.execute("SELECT * FROM enrollment_window_features").fetchdf()

# ------------------------------
# 3) Audits
# ------------------------------
audit_df = pd.DataFrame([{
    "table_name": "enrollment_window_features",
    "window_weeks": window_weeks,
    "n_rows": int(len(window_df)),
    "n_distinct_enrollments": int(window_df["enrollment_id"].nunique()) if len(window_df) > 0 else 0,
    "is_unique_by_enrollment_id": bool(
        len(window_df) == window_df["enrollment_id"].nunique()
    ) if len(window_df) > 0 else True
}])

missing_df = pd.DataFrame([{
    "n_missing_clicks": int(window_df[f"clicks_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_active_weeks": int(window_df[f"active_weeks_first_{window_weeks}"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_mean_clicks": int(window_df[f"mean_clicks_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_mean_distinct_sites": int(window_df[f"mean_distinct_sites_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
}])

# ------------------------------
# 4) Exports
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_window_features.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_window_features.csv"
audit_path = TABLES_DIR / "table_enrollment_window_features_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_window_features_missing_check.csv"

con.execute(f"COPY enrollment_window_features TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_window_features TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_df.to_csv(audit_path, index=False)
missing_df.to_csv(missing_path, index=False)

# ------------------------------
# 5) Output
# ------------------------------
print("\nEnrollment window feature audit:")
display(audit_df)

print("\nMissing check:")
display(missing_df)

print("\nSample rows:")
display(window_df.head(10))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())


B5.2 — Materialize Canonical Enrollment Window Features

Enrollment window feature audit:


,table_name,window_weeks,n_rows,n_distinct_enrollments,is_unique_by_enrollment_id
0,enrollment_window_features,4,32593,32593,True



Missing check:


,n_missing_clicks,n_missing_active_weeks,n_missing_mean_clicks,n_missing_mean_distinct_sites
0,0,0,0,0



Sample rows:


,enrollment_id,clicks_first_4_weeks,active_weeks_first_4,mean_clicks_first_4_weeks,mean_distinct_sites_first_4_weeks,max_week_observed_in_window
0,100561|DDD|2014J,107.0,3.0,26.75,4.25,3
1,1026777|FFF|2013B,150.0,3.0,37.50,6.75,3
2,105851|BBB|2013J,33.0,2.0,8.25,2.50,3
3,1076015|CCC|2014J,220.0,3.0,55.00,5.50,3
4,108497|EEE|2014J,311.0,4.0,77.75,8.25,3
5,1091274|CCC|2014J,155.0,4.0,38.75,7.25,3
6,111360|DDD|2013J,38.0,3.0,9.50,3.75,3
7,1118284|FFF|2013B,397.0,4.0,99.25,15.00,3
8,112245|DDD|2013B,432.0,4.0,108.00,23.50,3
9,117510|FFF|2014B,651.0,4.0,162.75,19.75,3



Saved:
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_window_features.parquet
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/data/enrollment_window_features.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_window_features_audit.csv
- /workspaces/survival_dropout_benchmark/outputs_benchmark_survival/tables/table_enrollment_window_features_missing_check.csv


: 

## B6 — Build Model-Specific Input Views (Revised)


In [ ]:
# ==============================================================
# B6 — Build Model-Specific Input Views (Revised)
# --------------------------------------------------------------
# Purpose:
#   Build model-specific input views aligned with the revised
#   benchmark configuration, including:
#     - person-period inputs for discrete-time models
#     - enrollment-level early-window comparable inputs for Cox
#       and DeepSurv
#
# Methodological note:
#   This revised version keeps the discrete-time inputs unchanged,
#   but refactors the enrollment-level model inputs so that Cox and
#   DeepSurv rely on:
#     - structural static covariates
#     - early-window temporal summaries
#
#   This avoids using retrospective whole-course temporal summaries
#   in the main enrollment-level benchmark track.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - DATA_OUTPUT_DIR
#   - MODEL_INPUT_VARIANTS
#   - MAIN_ENROLLMENT_WINDOW_WEEKS
#
# Outputs:
#   - pp_linear_hazard_input
#   - pp_neural_hazard_input
#   - enrollment_cox_input
#   - enrollment_deepsurv_input
#   - table_model_input_views_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("B6 — Build Model-Specific Input Views (Revised)")
print("=" * 70)
print("Methodological note: enrollment-level inputs now use the")
print("early-window comparable design instead of whole-course summaries.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "DATA_OUTPUT_DIR",
    "MODEL_INPUT_VARIANTS", "MAIN_ENROLLMENT_WINDOW_WEEKS"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun prior setup cells first."
    )

import pandas as pd
import numpy as np

# ------------------------------
# 2) Determine main window columns
# ------------------------------
if MAIN_ENROLLMENT_WINDOW_WEEKS == 4:
    main_window_cols = [
        "clicks_first_4_weeks",
        "active_weeks_first_4",
        "mean_clicks_first_4_weeks",
    ]
elif MAIN_ENROLLMENT_WINDOW_WEEKS == 2:
    main_window_cols = [
        "clicks_first_2_weeks",
        "active_weeks_first_2",
        "mean_clicks_first_2_weeks",
    ]
else:
    raise ValueError(
        f"Unsupported MAIN_ENROLLMENT_WINDOW_WEEKS={MAIN_ENROLLMENT_WINDOW_WEEKS} in B6."
    )

# ------------------------------
# 3) Discrete-time input views
# ------------------------------
con.execute("DROP TABLE IF EXISTS pp_linear_hazard_input")
con.execute("""
CREATE TABLE pp_linear_hazard_input AS
SELECT
    enrollment_id,
    id_student,
    code_module,
    code_presentation,
    week,
    event_t,
    event_observed,
    t_event_week,
    t_final_week,
    used_zero_week_fallback_for_censoring,
    gender,
    region,
    highest_education,
    imd_band,
    age_band,
    num_of_prev_attempts,
    studied_credits,
    disability,
    total_clicks_week,
    active_this_week,
    n_vle_rows_week,
    n_distinct_sites_week,
    cum_clicks_until_t,
    recency,
    streak
FROM person_period_enriched
""")

con.execute("DROP TABLE IF EXISTS pp_neural_hazard_input")
con.execute("""
CREATE TABLE pp_neural_hazard_input AS
SELECT * FROM pp_linear_hazard_input
""")

# ------------------------------
# 4) Enrollment-level early-window comparable inputs
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_cox_input")
con.execute(f"""
CREATE TABLE enrollment_cox_input AS
SELECT
    b.enrollment_id,
    b.id_student,
    b.code_module,
    b.code_presentation,
    b.event,
    b.duration,
    b.duration_raw,
    b.used_zero_week_fallback_for_censoring,
    b.gender,
    b.region,
    b.highest_education,
    b.imd_band,
    b.age_band,
    b.num_of_prev_attempts,
    b.studied_credits,
    b.disability,
    w.{main_window_cols[0]},
    w.{main_window_cols[1]},
    w.{main_window_cols[2]}
FROM enrollment_model_table b
LEFT JOIN enrollment_window_features w
    ON b.enrollment_id = w.enrollment_id
""")

con.execute("DROP TABLE IF EXISTS enrollment_deepsurv_input")
con.execute("""
CREATE TABLE enrollment_deepsurv_input AS
SELECT * FROM enrollment_cox_input
""")

# ------------------------------
# 5) Audit tables
# ------------------------------
audit_rows = []

for table_name, family, target_col, duration_col in [
    ("pp_linear_hazard_input", "discrete_time", "event_t", None),
    ("pp_neural_hazard_input", "discrete_time", "event_t", None),
    ("enrollment_cox_input", "continuous_time", "event", "duration"),
    ("enrollment_deepsurv_input", "continuous_time", "event", "duration"),
]:
    q = f"SELECT * FROM {table_name}"
    df = con.execute(q).fetchdf()

    row = {
        "table_name": table_name,
        "family": family,
        "n_rows": int(df.shape[0]),
        "n_distinct_enrollments": int(df['enrollment_id'].nunique()) if 'enrollment_id' in df.columns else np.nan,
        "n_columns": int(df.shape[1]),
        "target_column": target_col,
        "target_sum": float(df[target_col].sum()) if target_col in df.columns else np.nan,
        "duration_column": duration_col,
        "min_duration": float(df[duration_col].min()) if duration_col and duration_col in df.columns else np.nan,
        "max_duration": float(df[duration_col].max()) if duration_col and duration_col in df.columns else np.nan,
    }
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)

# sample tables
sample_pp_linear = con.execute("""
SELECT *
FROM pp_linear_hazard_input
ORDER BY enrollment_id, week
LIMIT 10
""").fetchdf()

sample_cox = con.execute("""
SELECT *
FROM enrollment_cox_input
ORDER BY enrollment_id
LIMIT 10
""").fetchdf()

# ------------------------------
# 6) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_model_input_views_audit.csv"
sample_pp_path = TABLES_DIR / "table_pp_linear_hazard_input_sample.csv"
sample_cox_path = TABLES_DIR / "table_enrollment_cox_input_sample.csv"

audit_df.to_csv(audit_path, index=False)
sample_pp_linear.to_csv(sample_pp_path, index=False)
sample_cox.to_csv(sample_cox_path, index=False)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nModel input views audit summary:")
display(audit_df)

print("\nSample — pp_linear_hazard_input:")
display(sample_pp_linear)

print("\nSample — enrollment_cox_input:")
display(sample_cox)

print("\nSaved audit:")
print("-", audit_path.resolve())
print("-", sample_pp_path.resolve())
print("-", sample_cox_path.resolve())


B6 — Build Model-Specific Input Views (Revised)
Methodological note: enrollment-level inputs now use the
early-window comparable design instead of whole-course summaries.


## B7 — Attach Canonical Window Features and Build Configured Inputs


In [ ]:
# ==============================================================
# B7 — Attach Canonical Window Features and Build Configured Inputs
# 
# ==============================================================
# Purpose:
#   Attach canonical enrollment-level window features driven by the
#   active configuration variables and add only the
#   non-duplicated columns to model-specific
#   enrollment-level inputs.
#
# Methodological note:
#   This version supports either:
#     - a scalar EARLY_WINDOW_WEEKS
#     - an iterable EARLY_WINDOW_WEEKS
#
#   It prevents duplicated columns when the base enrollment-level
#   inputs already contain the main comparable early-window
#   features.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - EARLY_WINDOW_WEEKS
#   - MODELING_CONFIG
#
# Outputs:
#   - DuckDB tables:
#       * enrollment_window_features
#       * enrollment_cox_input_configured
#       * enrollment_deepsurv_input_configured
#   - exported parquet/csv files
#   - table_enrollment_window_features_audit.csv
#   - table_enrollment_window_features_missing_check.csv
#   - table_enrollment_window_features_created_columns.csv
#   - table_enrollment_window_feature_attach_plan.csv
# ==============================================================

print("\n" + "=" * 70)
print("B7 — Build Window-Based Features and Experimental Variants")
print("=" * 70)
print("Methodological note: window-based features are generated from configuration variables.")
print("Convention adopted: first_w_weeks means rows where week < w.")
print("This version supports scalar or iterable EARLY_WINDOW_WEEKS.")
print("This version avoids duplicated columns in configured enrollment-level inputs.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR", "EARLY_WINDOW_WEEKS", "MODELING_CONFIG"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
import numpy as np
from pathlib import Path

DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUTPUT_DIR / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Validate and normalize windows
# ------------------------------
if isinstance(EARLY_WINDOW_WEEKS, (list, tuple, set)):
    WINDOWS = sorted(set(int(w) for w in EARLY_WINDOW_WEEKS))
else:
    WINDOWS = [int(EARLY_WINDOW_WEEKS)]

if len(WINDOWS) == 0:
    raise ValueError("No valid EARLY_WINDOW_WEEKS were provided.")

if any(w <= 0 for w in WINDOWS):
    raise ValueError(f"All EARLY_WINDOW_WEEKS must be positive integers. Got: {WINDOWS}")

print(f"\nConfigured EARLY_WINDOW_WEEKS: {WINDOWS}")


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 3) Helper utilities
# ------------------------------
def get_table_columns(table_name: str) -> list[str]:
    df_cols = con.execute(f"SELECT * FROM {table_name} LIMIT 0").df()
    return list(df_cols.columns)


### Funcao: duplicated_names

Definicao isolada de duplicated_names para reutilizacao nas etapas seguintes.


In [ ]:

def duplicated_names(cols: list[str]) -> list[str]:
    seen = set()
    dup = []
    for c in cols:
        if c in seen and c not in dup:
            dup.append(c)
        seen.add(c)
    return dup


### Funcao: build_window_select_fragment

Definicao isolada de build_window_select_fragment para reutilizacao nas etapas seguintes.


In [ ]:

def build_window_select_fragment(cols: list[str], alias: str = "ewf") -> str:
    if not cols:
        return ""
    return ",\n    " + ",\n    ".join(f"{alias}.{c}" for c in cols)


In [ ]:

# ------------------------------
# 4) Build dynamic SQL for window features
# ------------------------------
window_feature_sql_parts = []
created_feature_names = []

for w in WINDOWS:
    clicks_col = f"clicks_first_{w}_weeks"
    active_col = f"active_weeks_first_{w}"
    mean_col = f"mean_clicks_first_{w}_weeks"

    created_feature_names.extend([clicks_col, active_col, mean_col])

    window_feature_sql_parts.append(f"""
    COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) AS {clicks_col},
    COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) AS {active_col},
    CASE
        WHEN COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) > 0
        THEN
            COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) * 1.0
            / COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0)
        ELSE 0
    END AS {mean_col}
    """)

window_feature_sql = ",\n".join(window_feature_sql_parts)

# ------------------------------
# 5) Build enrollment_window_features
# ------------------------------
required_source_table = "person_period_enriched"
if required_source_table not in con.execute("SHOW TABLES").fetchdf()["name"].tolist():
    raise RuntimeError(
        "Required source table 'person_period_enriched' not found. "
        "Run prior preprocessing/person-period steps first."
    )

required_pp_cols = {
    "enrollment_id",
    "week",
    "total_clicks_week",
    "active_this_week",
}
pp_cols = set(get_table_columns(required_source_table))
missing_pp_cols = sorted(required_pp_cols - pp_cols)
if missing_pp_cols:
    raise KeyError(
        "person_period_enriched is missing required columns for B7: "
        + ", ".join(missing_pp_cols)
    )

con.execute("DROP TABLE IF EXISTS enrollment_window_features")
con.execute(f"""
CREATE TABLE enrollment_window_features AS
SELECT
    enrollment_id,
    {window_feature_sql}
FROM person_period_enriched
GROUP BY enrollment_id
""")

# ------------------------------
# 6) Discover base/input columns dynamically
# ------------------------------
for base_table in ["enrollment_cox_input", "enrollment_deepsurv_input"]:
    if base_table not in con.execute("SHOW TABLES").fetchdf()["name"].tolist():
        raise RuntimeError(
            f"Required table '{base_table}' not found. Run B6 first."
        )

cox_base_cols = get_table_columns("enrollment_cox_input")
deepsurv_base_cols = get_table_columns("enrollment_deepsurv_input")
window_cols = [c for c in get_table_columns("enrollment_window_features") if c != "enrollment_id"]

cox_extra_window_cols = [c for c in window_cols if c not in cox_base_cols]
deepsurv_extra_window_cols = [c for c in window_cols if c not in deepsurv_base_cols]

cox_window_fragment = build_window_select_fragment(cox_extra_window_cols, alias="ewf")
deepsurv_window_fragment = build_window_select_fragment(deepsurv_extra_window_cols, alias="ewf")

attach_plan_df = pd.DataFrame([
    {
        "target_table": "enrollment_cox_input_configured",
        "base_table": "enrollment_cox_input",
        "n_base_columns": len(cox_base_cols),
        "n_window_columns_available": len(window_cols),
        "n_window_columns_attached": len(cox_extra_window_cols),
        "attached_window_columns": ", ".join(cox_extra_window_cols),
        "n_window_columns_skipped_as_already_present": len([c for c in window_cols if c in cox_base_cols]),
        "skipped_window_columns": ", ".join([c for c in window_cols if c in cox_base_cols]),
    },
    {
        "target_table": "enrollment_deepsurv_input_configured",
        "base_table": "enrollment_deepsurv_input",
        "n_base_columns": len(deepsurv_base_cols),
        "n_window_columns_available": len(window_cols),
        "n_window_columns_attached": len(deepsurv_extra_window_cols),
        "attached_window_columns": ", ".join(deepsurv_extra_window_cols),
        "n_window_columns_skipped_as_already_present": len([c for c in window_cols if c in deepsurv_base_cols]),
        "skipped_window_columns": ", ".join([c for c in window_cols if c in deepsurv_base_cols]),
    },
])

# ------------------------------
# 7) Build configured Cox and DeepSurv inputs without duplication
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_cox_input_configured")
con.execute(f"""
CREATE TABLE enrollment_cox_input_configured AS
SELECT
    eci.*{cox_window_fragment}
FROM enrollment_cox_input AS eci
LEFT JOIN enrollment_window_features AS ewf
    ON eci.enrollment_id = ewf.enrollment_id
""")

con.execute("DROP TABLE IF EXISTS enrollment_deepsurv_input_configured")
con.execute(f"""
CREATE TABLE enrollment_deepsurv_input_configured AS
SELECT
    edi.*{deepsurv_window_fragment}
FROM enrollment_deepsurv_input AS edi
LEFT JOIN enrollment_window_features AS ewf
    ON edi.enrollment_id = ewf.enrollment_id
""")

# ------------------------------
# 8) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments
FROM enrollment_window_features
""").fetchdf()

audit_summary["n_windows_configured"] = len(WINDOWS)
audit_summary["window_list"] = [", ".join(str(w) for w in WINDOWS)]
audit_summary["n_created_features"] = len(created_feature_names)

# ------------------------------
# 9) Missing check for created features
# ------------------------------
missing_exprs = []
for feat in created_feature_names:
    missing_exprs.append(f"SUM(CASE WHEN {feat} IS NULL THEN 1 ELSE 0 END) AS n_missing_{feat}")

missing_sql = ",\n    ".join(missing_exprs)

missing_check = con.execute(f"""
SELECT
    {missing_sql}
FROM enrollment_window_features
""").fetchdf()

# ------------------------------
# 10) Duplicate safety checks
# ------------------------------
configured_cox_cols = get_table_columns("enrollment_cox_input_configured")
configured_deepsurv_cols = get_table_columns("enrollment_deepsurv_input_configured")

cox_dup_cols = duplicated_names(configured_cox_cols)
deepsurv_dup_cols = duplicated_names(configured_deepsurv_cols)

if cox_dup_cols:
    raise ValueError(f"Duplicated columns detected in enrollment_cox_input_configured: {cox_dup_cols}")

if deepsurv_dup_cols:
    raise ValueError(f"Duplicated columns detected in enrollment_deepsurv_input_configured: {deepsurv_dup_cols}")

suffixed_artifacts = [
    c for c in configured_cox_cols if c.endswith("_1")
] + [
    c for c in configured_deepsurv_cols if c.endswith("_1")
]
if suffixed_artifacts:
    raise ValueError(
        "Suffixed duplicate artifact columns detected after configured join: "
        + ", ".join(sorted(set(suffixed_artifacts)))
    )

# ------------------------------
# 11) Sample rows
# ------------------------------
sample_rows = con.execute("""
SELECT *
FROM enrollment_cox_input_configured
ORDER BY enrollment_id
LIMIT 20
""").fetchdf()

# ------------------------------
# 12) Export tables
# ------------------------------
export_tables = [
    "enrollment_window_features",
    "enrollment_cox_input_configured",
    "enrollment_deepsurv_input_configured",
]

for table_name in export_tables:
    parquet_path = DATA_OUTPUT_DIR / f"{table_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{table_name}.csv"
    con.execute(f"COPY {table_name} TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
    con.execute(f"COPY {table_name} TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_path = TABLES_DIR / "table_enrollment_window_features_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_window_features_missing_check.csv"
feature_list_path = TABLES_DIR / "table_enrollment_window_features_created_columns.csv"
attach_plan_path = TABLES_DIR / "table_enrollment_window_feature_attach_plan.csv"
sample_path = TABLES_DIR / "table_enrollment_cox_input_configured_sample.csv"

audit_summary.to_csv(audit_path, index=False)
missing_check.to_csv(missing_path, index=False)
pd.DataFrame({"created_feature_name": created_feature_names}).to_csv(feature_list_path, index=False)
attach_plan_df.to_csv(attach_plan_path, index=False)
sample_rows.to_csv(sample_path, index=False)

# ------------------------------
# 13) Output for feedback
# ------------------------------
print("\nWindow configuration used:")
print(" - EARLY_WINDOW_WEEKS:", WINDOWS)

print("\nCreated window-based feature columns:")
for feat in created_feature_names:
    print(" -", feat)

print("\nWindow attachment plan:")
display(attach_plan_df)

print("\nEnrollment window features audit summary:")
display(audit_summary)

print("\nMissing check for created features:")
display(missing_check)

print("\nSample — enrollment_cox_input_configured:")
display(sample_rows)

print("\nSaved:")
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", feature_list_path.resolve())
print("-", attach_plan_path.resolve())
print("-", sample_path.resolve())

print("\nExported tables:")
for table_name in export_tables:
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.parquet").resolve())
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.csv").resolve())


### B7.1 — Early-Window Length Sensitivity Design


In [ ]:
# ==============================================================
# B7.1 — Early-Window Length Sensitivity Design
# ==============================================================
# Purpose:
#   Build and audit a canonical early-window sensitivity design
#   across multiple window lengths, using enrollment-level
#   comparable inputs as the main structural target.
#
# Design:
#   - materialize window features for 2, 4, and 6 weeks
#   - build canonical comparable enrollment-level variants
#   - avoid duplicated columns
#
# Main outputs:
#   - enrollment_window_features_sensitivity
#   - enrollment_cox_input_w2
#   - enrollment_cox_input_w4
#   - enrollment_cox_input_w6
#   - enrollment_deepsurv_input_w2
#   - enrollment_deepsurv_input_w4
#   - enrollment_deepsurv_input_w6
#   - table_p12_1_window_sensitivity_design.csv
#   - table_p12_1_window_sensitivity_attach_plan.csv
#   - metadata_p12_1_window_sensitivity_design.json
# ==============================================================

print("\n" + "=" * 70)
print("B7.1 — Early-Window Length Sensitivity Design")
print("=" * 70)

required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd

OUT_BASE = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path(globals().get("SHARED_CONFIG", {}).get("paths", {}).get("output_dir", "outputs_benchmark_survival"))
DATA_OUTPUT_DIR = globals().get("DATA_OUTPUT_DIR", OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("data_output_subdir", "data"))
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS_SENSITIVITY = [2, 4, 6]

# --------------------------------------------------------------
# 1) Basic source checks
# --------------------------------------------------------------
available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())

required_tables = [
    "person_period_enriched",
    "enrollment_cox_input",
    "enrollment_deepsurv_input",
]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required tables for B7.1: " + ", ".join(missing_tables)
    )


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_table_columns(table_name: str) -> list[str]:
    return list(con.execute(f"SELECT * FROM {table_name} LIMIT 0").df().columns)


In [ ]:

pp_cols = set(get_table_columns("person_period_enriched"))
required_pp_cols = {"enrollment_id", "week", "total_clicks_week", "active_this_week"}
missing_pp_cols = sorted(required_pp_cols - pp_cols)
if missing_pp_cols:
    raise KeyError(
        "person_period_enriched is missing required columns for B7.1: "
        + ", ".join(missing_pp_cols)
    )

# --------------------------------------------------------------
# 2) Build full sensitivity window feature table
# --------------------------------------------------------------
window_feature_sql_parts = []
created_feature_names = []

for w in WINDOWS_SENSITIVITY:
    clicks_col = f"clicks_first_{w}_weeks"
    active_col = f"active_weeks_first_{w}"
    mean_col = f"mean_clicks_first_{w}_weeks"

    created_feature_names.extend([clicks_col, active_col, mean_col])

    window_feature_sql_parts.append(f"""
    COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) AS {clicks_col},
    COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) AS {active_col},
    CASE
        WHEN COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) > 0
        THEN
            COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) * 1.0
            / COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0)
        ELSE 0
    END AS {mean_col}
    """)

window_feature_sql = ",\n".join(window_feature_sql_parts)

con.execute("DROP TABLE IF EXISTS enrollment_window_features_sensitivity")
con.execute(f"""
CREATE TABLE enrollment_window_features_sensitivity AS
SELECT
    enrollment_id,
    {window_feature_sql}
FROM person_period_enriched
GROUP BY enrollment_id
""")


### Funcao: build_window_variant

Definicao isolada de build_window_variant para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 3) Materialize window-specific comparable variants
# --------------------------------------------------------------
def build_window_variant(base_table: str, out_table: str, window_w: int):
    base_cols = get_table_columns(base_table)

    target_cols = [
        f"clicks_first_{window_w}_weeks",
        f"active_weeks_first_{window_w}",
        f"mean_clicks_first_{window_w}_weeks",
    ]

    extra_cols = [c for c in target_cols if c not in base_cols]

    fragment = ""
    if extra_cols:
        fragment = ",\n    " + ",\n    ".join(f"ewf.{c}" for c in extra_cols)

    con.execute(f"DROP TABLE IF EXISTS {out_table}")
    con.execute(f"""
    CREATE TABLE {out_table} AS
    SELECT
        b.*{fragment}
    FROM {base_table} AS b
    LEFT JOIN enrollment_window_features_sensitivity AS ewf
        ON b.enrollment_id = ewf.enrollment_id
    """)

    final_cols = get_table_columns(out_table)
    duplicated = [c for i, c in enumerate(final_cols) if c in final_cols[:i]]
    if duplicated:
        raise ValueError(f"Duplicated columns detected in {out_table}: {duplicated}")

    return {
        "target_table": out_table,
        "base_table": base_table,
        "window_weeks": window_w,
        "n_base_columns": len(base_cols),
        "n_target_window_columns": len(target_cols),
        "n_window_columns_attached": len(extra_cols),
        "attached_window_columns": ", ".join(extra_cols),
        "n_window_columns_already_present": len([c for c in target_cols if c in base_cols]),
        "already_present_window_columns": ", ".join([c for c in target_cols if c in base_cols]),
    }


In [ ]:

attach_rows = []

for w in WINDOWS_SENSITIVITY:
    attach_rows.append(build_window_variant("enrollment_cox_input", f"enrollment_cox_input_w{w}", w))
    attach_rows.append(build_window_variant("enrollment_deepsurv_input", f"enrollment_deepsurv_input_w{w}", w))

table_p12_1_window_sensitivity_attach_plan = pd.DataFrame(attach_rows)

# --------------------------------------------------------------
# 4) Build design summary
# --------------------------------------------------------------
summary_rows = []

ewf_audit = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments
FROM enrollment_window_features_sensitivity
""").fetchdf().iloc[0]

for w in WINDOWS_SENSITIVITY:
    for family, table_name in [
        ("continuous_time_cox", f"enrollment_cox_input_w{w}"),
        ("continuous_time_deepsurv", f"enrollment_deepsurv_input_w{w}"),
    ]:
        df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
        summary_rows.append({
            "window_weeks": w,
            "family": family,
            "table_name": table_name,
            "n_rows": int(df.shape[0]),
            "n_distinct_enrollments": int(df["enrollment_id"].nunique()) if "enrollment_id" in df.columns else None,
            "n_columns": int(df.shape[1]),
            "has_clicks_feature": f"clicks_first_{w}_weeks" in df.columns,
            "has_active_feature": f"active_weeks_first_{w}" in df.columns,
            "has_mean_feature": f"mean_clicks_first_{w}_weeks" in df.columns,
        })

table_p12_1_window_sensitivity_design = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_design = TABLES_DIR / "table_p12_1_window_sensitivity_design.csv"
path_attach = TABLES_DIR / "table_p12_1_window_sensitivity_attach_plan.csv"
path_created = TABLES_DIR / "table_p12_1_window_sensitivity_created_columns.csv"
path_metadata = globals().get("METADATA_DIR", Path(OUTPUT_DIR) / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("metadata_subdir", "metadata")) / "metadata_p12_1_window_sensitivity_design.json"
path_metadata.parent.mkdir(parents=True, exist_ok=True)

table_p12_1_window_sensitivity_design.to_csv(path_design, index=False)
table_p12_1_window_sensitivity_attach_plan.to_csv(path_attach, index=False)
pd.DataFrame({"created_feature_name": created_feature_names}).to_csv(path_created, index=False)

metadata = {
    "step": "B7.1",
    "title": "Early-Window Length Sensitivity Design",
    "windows_sensitivity": WINDOWS_SENSITIVITY,
    "n_rows_window_feature_table": int(ewf_audit["n_rows"]),
    "n_distinct_enrollments_window_feature_table": int(ewf_audit["n_distinct_enrollments"]),
    "output_tables": [
        path_design.as_posix(),
        path_attach.as_posix(),
        path_created.as_posix(),
    ],
    "materialized_duckdb_tables": [
        "enrollment_window_features_sensitivity",
        "enrollment_cox_input_w2",
        "enrollment_cox_input_w4",
        "enrollment_cox_input_w6",
        "enrollment_deepsurv_input_w2",
        "enrollment_deepsurv_input_w4",
        "enrollment_deepsurv_input_w6",
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity windows used:")
print(WINDOWS_SENSITIVITY)

print("\nWindow sensitivity attach plan:")
display(table_p12_1_window_sensitivity_attach_plan)

print("\nWindow sensitivity design summary:")
display(table_p12_1_window_sensitivity_design)

print("\nSaved:")
print("-", path_design.resolve())
print("-", path_attach.resolve())
print("-", path_created.resolve())
print("-", path_metadata.resolve())


### B7.2 — Early-Window Length Sensitivity Comparison


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# B7.2 — Early-Window Length Sensitivity Comparison
# ==============================================================
# Purpose:
#   Consolidate the early-window sensitivity design into a
#   canonical comparison table for the feature-engineering section.
#
# Main outputs:
#   - table_p12_2_window_sensitivity_comparison.csv
#   - table_p12_2_window_sensitivity_summary.csv
#   - table_p12_2_window_sensitivity_registry.csv
#   - metadata_p12_2_window_sensitivity_comparison.json
# ==============================================================

print("\n" + "=" * 70)
print("B7.2 — Early-Window Length Sensitivity Comparison")
print("=" * 70)

required_names = ["con", "TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path(globals().get("SHARED_CONFIG", {}).get("paths", {}).get("output_dir", "outputs_benchmark_survival"))
OUT_METADATA = METADATA_DIR if "METADATA_DIR" in globals() else OUT_BASE / globals().get("SHARED_CONFIG", {}).get("paths", {}).get("metadata_subdir", "metadata")
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve current materialized sensitivity variants
# --------------------------------------------------------------
available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())

windows = [2, 4, 6]
families = [
    ("continuous_time_cox", "enrollment_cox_input_w{w}"),
    ("continuous_time_deepsurv", "enrollment_deepsurv_input_w{w}"),
]

registry_rows = []
comparison_rows = []

def get_table_columns(table_name: str) -> list[str]:
    return list(con.execute(f"SELECT * FROM {table_name} LIMIT 0").df().columns)

for family_name, template in families:
    for w in windows:
        table_name = template.format(w=w)
        exists = table_name in available_tables

        row = {
            "window_weeks": w,
            "family": family_name,
            "table_name": table_name,
            "table_exists": exists,
            "n_rows": np.nan,
            "n_distinct_enrollments": np.nan,
            "n_columns": np.nan,
            "has_clicks_feature": False,
            "has_active_feature": False,
            "has_mean_feature": False,
            "event_sum": np.nan,
            "duration_min": np.nan,
            "duration_max": np.nan,
            "ready_for_model_training": False,
            "notes": "",
        }

        if exists:
            df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
            cols = set(df.columns)

            clicks_col = f"clicks_first_{w}_weeks"
            active_col = f"active_weeks_first_{w}"
            mean_col = f"mean_clicks_first_{w}_weeks"

            row["n_rows"] = int(df.shape[0])
            row["n_distinct_enrollments"] = int(df["enrollment_id"].nunique()) if "enrollment_id" in df.columns else np.nan
            row["n_columns"] = int(df.shape[1])
            row["has_clicks_feature"] = clicks_col in cols
            row["has_active_feature"] = active_col in cols
            row["has_mean_feature"] = mean_col in cols

            if "event" in cols:
                row["event_sum"] = float(pd.to_numeric(df["event"], errors="coerce").fillna(0).sum())
            if "duration" in cols:
                row["duration_min"] = float(pd.to_numeric(df["duration"], errors="coerce").min())
                row["duration_max"] = float(pd.to_numeric(df["duration"], errors="coerce").max())

            row["ready_for_model_training"] = bool(
                row["has_clicks_feature"]
                and row["has_active_feature"]
                and row["has_mean_feature"]
                and ("event" in cols)
                and ("duration" in cols)
            )

            if w == 4:
                row["notes"] = "Main comparable early-window benchmark design."
            else:
                row["notes"] = "Sensitivity variant materialized and structurally ready."
        else:
            row["notes"] = "Variant table not materialized."

        registry_rows.append(row)

# canonical registry
table_p12_2_window_sensitivity_registry = (
    pd.DataFrame(registry_rows)
    .sort_values(["family", "window_weeks"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 2) Comparison table
# --------------------------------------------------------------
for family_name, g in table_p12_2_window_sensitivity_registry.groupby("family", dropna=False):
    main_row = g.loc[g["window_weeks"] == 4].copy()
    if len(main_row) == 0:
        for _, r in g.iterrows():
            comparison_rows.append({
                "family": family_name,
                "window_weeks": r["window_weeks"],
                "table_name": r["table_name"],
                "table_exists": r["table_exists"],
                "ready_for_model_training": r["ready_for_model_training"],
                "delta_n_columns_vs_w4": np.nan,
                "delta_event_sum_vs_w4": np.nan,
                "delta_duration_max_vs_w4": np.nan,
                "comparison_status": "missing_w4_reference",
                "notes": "No 4-week reference row available for comparison.",
            })
        continue

    main = main_row.iloc[0]

    for _, r in g.iterrows():
        comparison_rows.append({
            "family": family_name,
            "window_weeks": r["window_weeks"],
            "table_name": r["table_name"],
            "table_exists": r["table_exists"],
            "ready_for_model_training": r["ready_for_model_training"],
            "delta_n_columns_vs_w4": (
                pd.to_numeric(r["n_columns"], errors="coerce")
                - pd.to_numeric(main["n_columns"], errors="coerce")
            ) if pd.notna(r["n_columns"]) and pd.notna(main["n_columns"]) else np.nan,
            "delta_event_sum_vs_w4": (
                pd.to_numeric(r["event_sum"], errors="coerce")
                - pd.to_numeric(main["event_sum"], errors="coerce")
            ) if pd.notna(r["event_sum"]) and pd.notna(main["event_sum"]) else np.nan,
            "delta_duration_max_vs_w4": (
                pd.to_numeric(r["duration_max"], errors="coerce")
                - pd.to_numeric(main["duration_max"], errors="coerce")
            ) if pd.notna(r["duration_max"]) and pd.notna(main["duration_max"]) else np.nan,
            "comparison_status": (
                "reference_main_window"
                if r["window_weeks"] == 4 else
                "sensitivity_variant_ready"
                if bool(r["ready_for_model_training"]) else
                "not_ready"
            ),
            "notes": (
                "4-week reference window."
                if r["window_weeks"] == 4 else
                "Sensitivity variant can be used for downstream window-length comparison."
            ),
        })

table_p12_2_window_sensitivity_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["family", "window_weeks"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 3) Summary table
# --------------------------------------------------------------
summary_rows = []
for family_name, g in table_p12_2_window_sensitivity_registry.groupby("family", dropna=False):
    summary_rows.append({
        "family": family_name,
        "n_windows_expected": len(windows),
        "n_windows_materialized": int(pd.to_numeric(g["table_exists"], errors="coerce").fillna(False).sum()),
        "n_windows_ready_for_model_training": int(pd.to_numeric(g["ready_for_model_training"], errors="coerce").fillna(False).sum()),
        "all_windows_materialized": bool(g["table_exists"].all()),
        "all_windows_ready_for_model_training": bool(g["ready_for_model_training"].all()),
        "window_list_materialized": ", ".join(str(int(w)) for w in g.loc[g["table_exists"], "window_weeks"].tolist()),
    })

table_p12_2_window_sensitivity_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("family")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 4) Save outputs
# --------------------------------------------------------------
path_registry = TABLES_DIR / "table_p12_2_window_sensitivity_registry.csv"
path_comparison = TABLES_DIR / "table_p12_2_window_sensitivity_comparison.csv"
path_summary = TABLES_DIR / "table_p12_2_window_sensitivity_summary.csv"
path_metadata = OUT_METADATA / "metadata_p12_2_window_sensitivity_comparison.json"

table_p12_2_window_sensitivity_registry.to_csv(path_registry, index=False)
table_p12_2_window_sensitivity_comparison.to_csv(path_comparison, index=False)
table_p12_2_window_sensitivity_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "B7.2",
    "title": "Early-Window Length Sensitivity Comparison",
    "windows_evaluated_structurally": windows,
    "families": [f[0] for f in families],
    "purpose": (
        "Consolidate the currently materialized early-window sensitivity variants "
        "into a canonical comparison table for the feature-engineering section."
    ),
    "output_tables": [
        path_registry.as_posix(),
        path_comparison.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 5) Display
# --------------------------------------------------------------
print("\nWindow sensitivity registry:")
display(table_p12_2_window_sensitivity_registry)

print("\nWindow sensitivity comparison:")
display(table_p12_2_window_sensitivity_comparison)

print("\nWindow sensitivity summary:")
display(table_p12_2_window_sensitivity_summary)

print("\nSaved:")
print("-", path_registry.resolve())
print("-", path_comparison.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())